# Instructor run (prepared before the session)

For the workshop team, not attendees. **Same `octank` package, same steps and settings as the
attendee notebooks** - only the dataset is 20x larger:

1. **Data build** - 20,000 raw requests -> curation -> gap-fill -> teacher traces -> SFT / RLVR / eval
   files (`scripts/build_data.py instructor`).
2. **Train and evaluate** - SFT, then SFT -> RLVR with the same reward Lambda and hyperparameters
   (`scripts/train_eval.py instructor`).
3. **Final evaluation** of every demo model in one pass, which also writes
   `data/reference_results.csv` for Lab 5 (`scripts/final_eval.py`).
4. **Demo endpoint** with the instructor and attendee-scale adapters (`scripts/deploy_demo.py`).

Seed data (catalog, gold set, benchmarks, scenarios) is committed in `data/`. The run's LLM caches and
state are version-controlled in `instructor/`, so a rebuild only calls Bedrock for inputs that changed;
derived files (parquet, datasets, logs) go to `work/instructor/`. The long steps run as background
scripts; this notebook monitors them.

In [ ]:
# Pinned SDKs (latest as of 2026-10-28). Restart the kernel after the first install.
%pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("instructor")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

## 1. Data build (background, resumable)

In [ ]:
STATUS = S.local("logs", "status.json")
# !cd .. && nohup ~/.venvs/gift_concierge/bin/python scripts/build_data.py instructor > work/instructor/logs/build.out 2>&1 &
print(open(STATUS).read() if STATUS.exists() else "not started")

In [ ]:
from octank import profile
from octank.pipeline import INSTRUCTOR, Run
run = Run(S, INSTRUCTOR)
raw = pd.read_parquet(run.path("raw_requests.parquet"))
final = pd.read_parquet(run.path("final.parquet"))
print(len(raw), "raw ->", len(final), "final")
pd.read_csv(run.path("curation_report.csv"))

In [ ]:
print("raw skew:", profile.skew_report(raw), "\nfinal skew:", profile.skew_report(final))
profile.plot_overview(final, "Instructor dataset (final)");

## 2. Gold set and human annotation slice

`data/gold_labels.jsonl` (150 rows, `gpt-oss-120b` drafts) calibrates the labeler in Lab 2.
`data/human_annotation_slice.csv` (50 rows) has the message, the `draft_*` labels and empty `human_*`
columns: annotators fill in `human_*` only where the draft is wrong, plus `annotator`. Merge it back:

In [ ]:
from octank import gold
g = gold.load_gold()
print(len(g), g["_flaw"].value_counts().to_dict())
# g = gold.merge_human(g, gold.HUMAN_SLICE_PATH.with_name("human_annotation_slice_done.csv"))
# gold.save_gold(g)
pd.read_csv(gold.HUMAN_SLICE_PATH).head()

## 3. Train and evaluate: SFT, then SFT -> RLVR

```bash
cd .. && nohup ~/.venvs/gift_concierge/bin/python scripts/train_eval.py instructor > work/instructor/logs/train_eval.out 2>&1 &
```
Same arms as the attendee Lab 3 (`octank.arms`), with evaluation on (`evaluate=True`); resumable from `instructor/state.json`.

In [ ]:
from octank import arms, workflow
arms.status(S)

In [ ]:
res = arms.results(S)
workflow.by_problem_type(res).pivot(index="problem_type", columns="model", values="rule_pass").round(3) if len(res) else "no results yet"

## 4. Final evaluation and demo endpoint

```bash
cd .. && nohup ~/.venvs/gift_concierge/bin/python scripts/final_eval.py > work/final_eval/run.out 2>&1 &
cd .. && nohup ~/.venvs/gift_concierge/bin/python scripts/deploy_demo.py > work/demo/deploy.out 2>&1 &
```
`final_eval.py` scores base, instructor and attendee-scale models in one pass (same cases) and
refreshes `data/reference_results.csv`.